In [1]:
import pickle 
import numpy as np 
import itertools
import pandas as pd
from pathlib import Path
import os

### Make manifest for alternate architectures

In [2]:
archs_to_skip = [0, 3, 5] # arch configs that didn't train (too large, didn't converge, etc.)
path_to_arch_configs = [config for config in Path("config/arch_search").glob('*v10*.yaml') 
                        if not any([f'_{i}' in config.stem for i in archs_to_skip])]

path_to_arch_configs += [config for config in Path("config/binaural_attn").glob('*v10*.yaml') 
                        if not 'cue_loc' in config.stem]
### Get checkpoints 
# get latest checkpoint for each config 
ckpt_dir = Path('attn_cue_models')

ckpt_dict = {}
for config in path_to_arch_configs:
    ckpt_files = list(ckpt_dir.glob(f'{config.stem}/checkpoints/*.ckpt'))
    if len(ckpt_files) == 0:
        print(f"Warning: no ckpt found for {config.stem}")
        continue
    latest_ckpt = max(ckpt_files, key=lambda x: x.stat().st_mtime)
    ckpt_dict[config.stem] = latest_ckpt

## Get good configs 
path_to_arch_configs = [config for config in path_to_arch_configs if config.stem in ckpt_dict]

print("N archs to test:", len(path_to_arch_configs))
print("Configs (as posix paths)")
print("\n".join( str(path) for path in path_to_arch_configs))


N archs to test: 13
Configs (as posix paths)
config/arch_search/word_task_v10_4MGB_ln_first_arch_1.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_10.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_12.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_2.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_4.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_6.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_7.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_8.yaml
config/arch_search/word_task_v10_4MGB_ln_first_arch_9.yaml
config/binaural_attn/word_task_early_only_v10.yaml
config/binaural_attn/word_task_late_only_v10.yaml
config/binaural_attn/word_task_v10_control_no_attn.yaml
config/binaural_attn/word_task_v10_main_feature_gain_config.yaml


In [3]:
ckpt_dict
parent_dir = Path('attn_cue_models')
for k,v in ckpt_dict.items():
    v = Path(v)
    full_path = parent_dir / f"{k}/{v.parent.stem}/{v.name}"
    if not full_path.exists():
        print(f"Warning: {full_path} does not exist") 
    print(f"{k}/{v.parent.stem}/{v.name}")

word_task_v10_4MGB_ln_first_arch_1/checkpoints/epoch=2-step=44750-v5.ckpt
word_task_v10_4MGB_ln_first_arch_10/checkpoints/epoch=2-step=33358-v6.ckpt
word_task_v10_4MGB_ln_first_arch_12/checkpoints/epoch=2-step=33358-v5.ckpt
word_task_v10_4MGB_ln_first_arch_2/checkpoints/epoch=3-step=50037-v3.ckpt
word_task_v10_4MGB_ln_first_arch_4/checkpoints/epoch=1-step=34375-v4.ckpt
word_task_v10_4MGB_ln_first_arch_6/checkpoints/epoch=0-step=8000-v1.ckpt
word_task_v10_4MGB_ln_first_arch_7/checkpoints/epoch=0-step=8000-v3.ckpt
word_task_v10_4MGB_ln_first_arch_8/checkpoints/epoch=4-step=54716-v4.ckpt
word_task_v10_4MGB_ln_first_arch_9/checkpoints/epoch=2-step=37358-v1.ckpt
word_task_early_only_v10/checkpoints/epoch=7-step=92753.ckpt
word_task_late_only_v10/checkpoints/epoch=7-step=96753.ckpt
word_task_v10_control_no_attn/checkpoints/epoch=7-step=94753.ckpt
word_task_v10_main_feature_gain_config/checkpoints/epoch=3-step=48037.ckpt


In [6]:
test_condition_manifest = {}
for ix, config in enumerate(path_to_arch_configs):
    
    test_condition_manifest[ix] = {
        'config_path': config,
        'ckpt_path': ckpt_dict[config.stem],
   }



In [7]:
outdir = Path("model_architecture_activation_manifests")
outdir.mkdir(exist_ok=True, parents=True)

# with open(outdir / "all_v10_architectures_alts_and_controls.pkl", 'wb') as f:
#     pickle.dump(test_condition_manifest, f, protocol=pickle.HIGHEST_PROTOCOL)
#     # match_human_pilot_conds = pickle.load(f)

In [8]:
isinstance(test_condition_manifest[1], dict)

True

In [18]:
len(test_condition_manifest)

13

In [19]:
manifest = pd.read_pickle("model_architecture_activation_manifests/all_v10_architectures_alts_and_controls.pkl")

In [20]:
manifest

{0: {'config_path': PosixPath('config/arch_search/word_task_v10_4MGB_ln_first_arch_1.yaml'),
  'ckpt_path': PosixPath('attn_cue_models/word_task_v10_4MGB_ln_first_arch_1/checkpoints/epoch=2-step=44750-v5.ckpt')},
 1: {'config_path': PosixPath('config/arch_search/word_task_v10_4MGB_ln_first_arch_10.yaml'),
  'ckpt_path': PosixPath('attn_cue_models/word_task_v10_4MGB_ln_first_arch_10/checkpoints/epoch=2-step=33358-v6.ckpt')},
 2: {'config_path': PosixPath('config/arch_search/word_task_v10_4MGB_ln_first_arch_12.yaml'),
  'ckpt_path': PosixPath('attn_cue_models/word_task_v10_4MGB_ln_first_arch_12/checkpoints/epoch=2-step=33358-v5.ckpt')},
 3: {'config_path': PosixPath('config/arch_search/word_task_v10_4MGB_ln_first_arch_2.yaml'),
  'ckpt_path': PosixPath('attn_cue_models/word_task_v10_4MGB_ln_first_arch_2/checkpoints/epoch=3-step=50037-v3.ckpt')},
 4: {'config_path': PosixPath('config/arch_search/word_task_v10_4MGB_ln_first_arch_4.yaml'),
  'ckpt_path': PosixPath('attn_cue_models/word_task

In [7]:
## Get keys from test_condition_manifest if arch_4 arch_9 arch_6 arch_12

ixs_to_test = []
for k,v in test_condition_manifest.items():
    if 'arch_2' in v['config_path'].stem:
        ixs_to_test.append(k)
    # if 'arch_9' in v['config_path'].stem:
    #     ixs_to_test.append(k)
    # if 'arch_6' in v['config_path'].stem:
    #     ixs_to_test.append(k)
    # if 'arch_12' in v['config_path'].stem:
    #     ixs_to_test.append(k)

# Get start and end of each consecutive sequence of ixs_to_test
ixs_to_test.sort()
ixs_to_test = np.array(ixs_to_test)
ixs_to_test_diff = np.diff(ixs_to_test)
ixs_to_test_diff = np.insert(ixs_to_test_diff, 0, 1)
ixs_to_test_diff = np.append(ixs_to_test_diff, 1)



In [8]:
ixs_to_test
# print 

array([183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195,
       196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208,
       209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,
       222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234,
       235, 236, 237, 238, 239, 240, 241, 242, 243])

In [7]:
### Print n tests for job array ix size 

n_tests = len(test_condition_manifest)
print(f"N tests: {n_tests}")
print(f"Array_ids: 0-{n_tests-1}")


N tests: 549
Array_ids: 0-548
